## Helper Function to load HDF5 

In [2]:
from pathlib import Path
import h5py
import time

def load_frames(path, stop): 
    with h5py.File(path, "r") as f:
            dset = f["rf"] if "rf" in f else f[f["RcvData"][0, 0]]
            # rest are zeros
            n_useful = 128 * 1920
            # Get size in bytes
            file_size_bytes = path.stat().st_size / (1024**3)
            
            print("\npath:", path)
            print(f"size: {file_size_bytes:.2f}. GB")
            print("shape:", dset.shape)
            print("chunks:", dset.chunks)
            print("compression:", dset.compression)
            t0 = time.perf_counter()
            x = dset[:stop, :, :n_useful]
            print("load time:", time.perf_counter() - t0, "s")
            print("loaded shape:", x.shape)

## Define Paths

In [17]:
filename = "115601.mat"
n_frames = 10
# fast SSD-backed storage in /work
work_dir = Path("/work/users/t/i/tis/data/jhu_spatiotemporal/")
# slower Storage /proj
proj_dir = Path("/proj/yzlinlab/projects/jhu_spatiotemporal/")

paths_dict = {
    "old_chunksize_proj_gzip": proj_dir / "data260421" / filename,
    "old_chunksize_work_gzip": work_dir / "data260421" / filename,
    "work_gzip": work_dir / "fastdata260421_gzip" / filename, 
    "work_lzf": work_dir / "fastdata260421_lzf" / filename,
    "work_no_compression": work_dir / "fastdata260421_no_compression" / filename,
}

## Transform HDF5 File

In [4]:
# Different Copression Algorithms supported by HDF5
compression_type = "gzip"
#compression_type = "lzf"
#compression_type = None

src_path = paths_dict["old_chunksize_work_gzip"]
dst_path = base_dir / "fastdata260421_lzf" / filename
dst_path.parent.mkdir(parents=True, exist_ok=True)

n_frames = 100
n_rx = 128
n_tx = 128
samples_per_tx = 1920
n_useful = n_tx * samples_per_tx

with h5py.File(src_path, "r") as src, h5py.File(dst_path, "w") as dst:
    ref = src["RcvData"][0, 0]
    src_dset = src[ref]

    out = dst.create_dataset(
        "rf",
        shape=(n_frames, n_rx, n_useful),
        dtype=src_dset.dtype,
        chunks=(1, n_rx, n_useful),
        compression=compression_type,
    )

    t0 = time.perf_counter()
    data = src_dset[:, :, :n_useful]
    print(f"read source in {time.perf_counter() - t0:.2f}s")

    t0 = time.perf_counter()
    out[:, :, :] = data
    print(f"wrote output in {time.perf_counter() - t0:.2f}s")
    # free up memory to avoid OOM
    del data

read source in 63.55s
wrote output in 189.24s


## Experiments of HDF5 Load Times

### Load Times /work vs /proj 

In [12]:
load_frames(paths_dict["old_chunksize_proj_gzip"], n_frames)
load_frames(paths_dict["old_chunksize_work_gzip"], n_frames)


path: /proj/yzlinlab/projects/jhu_spatiotemporal/data260421/115601.mat
size: 3.62. GB
shape: (100, 128, 524288)
chunks: (100, 128, 2)
compression: gzip
load time: 36.820940643548965 s
loaded shape: (10, 128, 245760)

path: /work/users/t/i/tis/data/jhu_spatiotemporal/data260421/115601.mat
size: 3.62. GB
shape: (100, 128, 524288)
chunks: (100, 128, 2)
compression: gzip
load time: 35.43086827173829 s
loaded shape: (10, 128, 245760)


Because of the unfortunate chunksize the dataloading is completely CPU bound

### Load Times Old vs. New Chunksize in /work

In [11]:
load_frames(paths_dict["old_chunksize_work_gzip"], n_frames)
load_frames(paths_dict["work_gzip"], n_frames)


path: /work/users/t/i/tis/data/jhu_spatiotemporal/data260421/115601.mat
size: 3.62. GB
shape: (100, 128, 524288)
chunks: (100, 128, 2)
compression: gzip
load time: 35.10455506667495 s
loaded shape: (10, 128, 245760)

path: /work/users/t/i/tis/data/jhu_spatiotemporal/fastdata260421_gzip/115601.mat
size: 3.52. GB
shape: (100, 128, 245760)
chunks: (1, 128, 245760)
compression: gzip
load time: 3.357792219147086 s
loaded shape: (10, 128, 245760)


### Load Times using different compression / no compression

In [18]:
load_frames(paths_dict["work_lzf"], n_frames)
load_frames(paths_dict["work_no_compression"], n_frames)


path: /work/users/t/i/tis/data/jhu_spatiotemporal/fastdata260421_lzf/115601.mat
size: 5.08. GB
shape: (100, 128, 245760)
chunks: (1, 128, 245760)
compression: lzf
load time: 2.0180099569261074 s
loaded shape: (10, 128, 245760)

path: /work/users/t/i/tis/data/jhu_spatiotemporal/fastdata260421_no_compression/115601.mat
size: 5.86. GB
shape: (100, 128, 245760)
chunks: (1, 128, 245760)
compression: None
load time: 1.1276805214583874 s
loaded shape: (10, 128, 245760)
